# PyTorch Training Loops

**PyTorch Interview Preparation · Notebook 06 · Estimated study time: 2–3 hours**

This is the notebook to revisit until the basic training and validation loops feel automatic. The core study cycle is:

```text
look at a blank loop → write it → run it → inspect metrics → debug it → write it again
```

Difficulty labels: 🟢 Easy · 🟡 Medium · 🔴 Hard


## 1. Setup

Everything is synthetic, offline, CPU-safe, and deliberately small.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)


## 2. What Is a Training Loop?

A training loop repeatedly passes batches through the model, computes loss, calculates gradients, and updates parameters.

```text
Batch → Model → Prediction → Loss → Backward → Gradients → Optimizer Step → Updated Model
  ↑__________________________________________________________________________________|
```

The next batch uses the updated parameters.


## 3. Epochs and Batches — 🔥 Interview Essential · 🟢 Easy

```text
Dataset = all training examples
Batch   = a subset processed together
Epoch   = one complete pass through the training dataset
```

With 1,000 samples and batch size 100 there are about 10 batches per epoch. A final batch can be smaller when the dataset size is not divisible by the batch size.

**Quick recall**

1. Dataset = 10,000, batch size = 250. How many full batches? **40**.
2. Dataset = 1,025, batch size = 100. How many batches including the remainder? **11**.
3. If a loader has 16 batches, how many parameter-update opportunities occur in one ordinary epoch? **16**.
4. Does three epochs mean each sample is necessarily seen in the same order? **No**, not with shuffling.


## 4. Small Synthetic Dataset and Model

`DataLoader` creates mini-batches; its details belong in Notebook 07. Here we only need to iterate over it. The primary task is multiclass classification.


In [ ]:
torch.manual_seed(42)

X = torch.randn(500, 10)
true_w = torch.randn(10, 3)
logits = X @ true_w
y = logits.argmax(dim=1)

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

X_batch, y_batch = next(iter(train_loader))
print("batch:", X_batch.shape, y_batch.shape)
print("number of batches:", len(train_loader))


## 5. The Core Training Step — 🔥 Interview Essential

Write this order from memory:

```python
optimizer.zero_grad()          # clear gradients from the previous step
logits = model(X_batch)        # forward pass
loss = loss_fn(logits, y_batch)# calculate error
loss.backward()                # calculate gradients
optimizer.step()               # update parameters
```

The forward pass builds the computation graph. `backward()` fills parameter `.grad` fields. `step()` reads those gradients. Reversing this dependency order breaks learning.


In [ ]:
# A real, automatically checked training step
before = [p.detach().clone() for p in model.parameters()]

optimizer.zero_grad()
logits = model(X_batch)
loss = loss_fn(logits, y_batch)
loss.backward()

assert loss.ndim == 0
assert any(p.grad is not None for p in model.parameters())

optimizer.step()
after = list(model.parameters())
assert any(not torch.equal(a, b) for a, b in zip(before, after))
print(f"step loss: {loss.item():.4f}; parameters changed ✓")


### Training-order exercises — 🔥 Interview Essential

Reorder each before revealing the answer.

1. `backward → step → forward → zero_grad → loss`
2. `forward → zero_grad → loss → step → backward`
3. `loss → forward → zero_grad → backward → step`

<details><summary>Solutions</summary>

All three reduce to: `zero_grad → forward → loss → backward → step`.

</details>

### Why `zero_grad()` comes first

PyTorch accumulates gradients by default: without clearing, `old gradients + new gradients` are stored. That is useful only when accumulation is intentional. If gradients grow every ordinary batch, first inspect whether `zero_grad()` is missing.


## 6. Training Mode — `model.train()` — 🔥 Interview Essential

`model.train()` puts modules into training behavior. It **does not start training automatically**. Dropout becomes active and BatchNorm uses batch behavior; actual learning still requires forward, loss, backward, and optimizer steps.

> **Interview answer:** `model.train()` sets training behavior for modules such as Dropout and BatchNorm; training itself still happens through forward, backward, and optimizer steps.


## 7. Full One-Epoch Training Loop

Before revealing this, write it on paper from memory.


In [ ]:
model.train()

for X_batch, y_batch in train_loader:
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = loss_fn(logits, y_batch)
    loss.backward()
    optimizer.step()

print("one epoch complete; model.training =", model.training)


## 8. `loss.item()` and Tracking Training Loss

`loss` is a scalar tensor connected to the graph. `loss.item()` returns a regular Python number for logging and metric accumulation.

```python
print(loss)        # tensor(..., grad_fn=...)
print(loss.item()) # Python float
```

Do not replace the tensor before backpropagation: `loss.item().backward()` cannot work because a Python float has neither a graph nor `backward()`.

A simple batch average is `sum(batch_losses) / len(loader)`. A sample-weighted average is more robust when the last batch is smaller.


## 9. Correct Loss Averaging — 🔥 Interview Essential · 🟡 Medium

Most PyTorch losses default to `reduction="mean"`, so `loss.item()` is the mean for that batch. Multiply it by batch size to recover that batch's total contribution:

```python
total_loss += loss.item() * batch_size
total_samples += batch_size
average_loss = total_loss / total_samples
```


## 10. Classification Accuracy — 🔥 Interview Essential

```text
logits (batch, classes) → argmax(dim=1) → predictions (batch,)
targets (batch,)        → elementwise comparison → count correct
```

Divide correct predictions by **samples**, not batches.


In [ ]:
with torch.no_grad():
    demo_logits = model(X_batch)
    predictions = demo_logits.argmax(dim=1)
    correct = (predictions == y_batch).sum().item()

assert predictions.shape == y_batch.shape
assert 0 <= correct <= y_batch.size(0)
print("correct in batch:", correct, "/", y_batch.size(0))


## 11. Complete Training Metrics Loop — 🔥 Interview Essential

Metrics are recorded after the update, but these logits came from the forward pass immediately before that update. That is the normal inexpensive convention.


In [ ]:
model.train()
total_loss = 0.0
total_correct = 0
total_samples = 0

for X_batch, y_batch in train_loader:
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = loss_fn(logits, y_batch)
    loss.backward()
    optimizer.step()

    batch_size = X_batch.size(0)
    total_loss += loss.item() * batch_size
    predictions = logits.argmax(dim=1)
    total_correct += (predictions == y_batch).sum().item()
    total_samples += batch_size

train_loss = total_loss / total_samples
train_accuracy = total_correct / total_samples
assert isinstance(train_loss, float)
assert 0 <= train_accuracy <= 1
print(f"train loss={train_loss:.4f}, accuracy={train_accuracy:.3f}")


## 12. Evaluation Loop — 🔥 Interview Essential

```python
model.eval()
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
```

Validation has **no** `zero_grad()`, **no** `backward()`, and **no** `step()`. Validation data measures generalization; it must not update the model.

| Training | Validation |
|---|---|
| `model.train()` | `model.eval()` |
| forward + loss | forward + loss |
| backward + update | no backward, no update |
| gradients needed | gradients not needed |
| Dropout active | Dropout disabled |
| BatchNorm batch behavior | BatchNorm evaluation behavior |


## 13. `model.eval()` vs `torch.no_grad()` — 🔥 Very Common Interview Question

`model.eval()` changes module behavior. `torch.no_grad()` disables gradient tracking. Neither replaces the other.

```text
model.eval()    → Dropout / BatchNorm evaluation behavior
torch.no_grad() → no Autograd graph, less memory and computation
```

`model.eval()` alone does **not** disable gradients. Validation normally uses both.


In [ ]:
probe = nn.Sequential(nn.Linear(10, 10), nn.Dropout(0.9))
probe.eval()
out = probe(torch.randn(2, 10))
assert out.requires_grad  # eval mode did not disable Autograd

with torch.no_grad():
    out_no_grad = probe(torch.randn(2, 10))
assert not out_no_grad.requires_grad
print("eval changes mode; no_grad changes gradient tracking ✓")


## 14. Reusable `train_one_epoch()` and `evaluate()`

These are the two functions to reproduce in interviews. Both use sample-weighted loss and sample-level accuracy.


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


### Automatic function validation

We verify parameter updates, Python metric types, valid accuracy, evaluation mode, and no validation update.


In [ ]:
device = torch.device("cpu")
check_model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 3)).to(device)
check_optimizer = optim.AdamW(check_model.parameters(), lr=1e-3)

before_train = [p.detach().clone() for p in check_model.parameters()]
check_loss, check_acc = train_one_epoch(
    check_model, train_loader, loss_fn, check_optimizer, device
)
after_train = [p.detach().clone() for p in check_model.parameters()]
assert any(not torch.equal(a, b) for a, b in zip(before_train, after_train))
assert isinstance(check_loss, float) and isinstance(check_acc, float)
assert 0 <= check_acc <= 1

before_eval = [p.detach().clone() for p in check_model.parameters()]
eval_loss, eval_acc = evaluate(check_model, train_loader, loss_fn, device)
after_eval = [p.detach().clone() for p in check_model.parameters()]
assert all(torch.equal(a, b) for a, b in zip(before_eval, after_eval))
assert not check_model.training and 0 <= eval_acc <= 1
print("train_one_epoch and evaluate checks passed ✓")


## 15. Device Handling — 🔥 Interview Essential · 🟡 Medium

Model parameters and input tensors must generally share a device. Moving only the model to CUDA while leaving input on CPU causes a device mismatch.

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)
```

Optional Apple Silicon support:


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)
print("selected device:", device)


## 16. Multiple Epochs, Logging, and History

The inner loop processes batches; the outer loop repeats epochs.

```text
Epoch 1 ├─ Batch 1 ├─ Batch 2 └─ ...
Epoch 2 ├─ Batch 1 ├─ Batch 2 └─ ...
```


In [ ]:
# A fresh reproducible train/validation split
torch.manual_seed(42)
indices = torch.randperm(len(dataset))
train_idx, val_idx = indices[:400], indices[400:]
train_ds = TensorDataset(X[train_idx], y[train_idx])
val_ds = TensorDataset(X[val_idx], y[val_idx])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-2)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
num_epochs = 8

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

assert all(len(values) == num_epochs for values in history.values())
assert history["train_loss"][-1] < history["train_loss"][0]


## 17. Learning Curves · 🟡 Medium

- Healthy: train loss ↓ and validation loss ↓.
- Overfitting: train loss continues ↓ while validation loss begins ↑.
- Underfitting: both losses remain high.
- Possible unstable learning rate: loss oscillates strongly or explodes.

Do not make a strong diagnosis from one metric alone.


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, num_epochs + 1)
plt.figure(figsize=(6, 3.5))
plt.plot(epochs, history["train_loss"], marker="o", label="train")
plt.plot(epochs, history["val_loss"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Learning curves")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 18. Gradient Clipping — 🔥 Interview Useful · 🟡 Medium

Gradient clipping limits gradient magnitude after gradients exist and before the update:

```python
optimizer.zero_grad()
loss = loss_fn(model(X_batch), y_batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

It is commonly useful for exploding gradients, RNNs, and some transformer workloads. **Order:** after `backward()`, before `step()`.


In [ ]:
# Runnable clipped step
X_batch, y_batch = next(iter(train_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)
model.train()
optimizer.zero_grad()
clip_loss = loss_fn(model(X_batch), y_batch)
clip_loss.backward()
pre_clip_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
assert torch.isfinite(pre_clip_norm)
print("pre-clipping total norm:", round(pre_clip_norm.item(), 4))


## 19. Gradient Accumulation — 🟡 Medium / 🔴 Robust Edge Cases

Accumulation simulates a larger effective batch when memory cannot hold it at once. With physical batch 16 and four accumulation steps, effective batch is approximately 64 on one device.

Dividing loss by accumulation steps keeps gradient scale approximately comparable to one larger averaged batch. A robust loop must also step for the final partial window.


In [ ]:
accumulation_steps = 4
accum_model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 3)).to(device)
accum_optimizer = optim.AdamW(accum_model.parameters(), lr=1e-3)
accum_model.train()
accum_optimizer.zero_grad()

for step, (X_batch, y_batch) in enumerate(train_loader):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    loss = loss_fn(accum_model(X_batch), y_batch) / accumulation_steps
    loss.backward()

    is_boundary = (step + 1) % accumulation_steps == 0
    is_last = step + 1 == len(train_loader)
    if is_boundary or is_last:
        accum_optimizer.step()
        accum_optimizer.zero_grad()

print("gradient accumulation epoch complete ✓")


**Effective-batch quick recall**

1. Physical 8 × accumulation 4 = **approximately 32**.
2. Physical 32 × accumulation 8 = **approximately 256**.
3. Physical 24 × accumulation 3 = **approximately 72**.


## 20. Early Stopping — 🔴 Hard to Make Robust

Early stopping halts training when validation performance stops improving for a chosen **patience**. Real systems usually save and restore the best checkpoint.


In [ ]:
# Concept-only simulation; no files are written.
example_val_losses = [0.80, 0.65, 0.61, 0.62, 0.64, 0.66]
best_val_loss = float("inf")
patience = 3
bad_epochs = 0
stopped_at = None

for epoch, val_loss in enumerate(example_val_losses, start=1):
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bad_epochs = 0
    else:
        bad_epochs += 1
    if bad_epochs >= patience:
        stopped_at = epoch
        break

assert stopped_at == 6
print("simulated early stop at epoch", stopped_at)


## 21. Reproducibility

`torch.manual_seed(42)` makes PyTorch random-number generation more reproducible. With CUDA, also use `torch.cuda.manual_seed_all(42)`. Exact reproducibility can still depend on hardware, algorithms, data-loader workers, and library versions; a seed is not a guarantee of universal bitwise determinism.


In [ ]:
torch.manual_seed(42)
first = torch.randn(3)
torch.manual_seed(42)
second = torch.randn(3)
assert torch.equal(first, second)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print("seeded draws match ✓")


## 22. Common Task Patterns

| Task | Loss | Prediction |
|---|---|---|
| Regression | `MSELoss` | raw model output |
| Binary | `BCEWithLogitsLoss` | sigmoid + threshold |
| Multiclass | `CrossEntropyLoss` | `argmax(dim=1)` |


In [ ]:
# Binary classification: logits and float targets use shape (batch, 1)
binary_model = nn.Linear(5, 1)
binary_X = torch.randn(12, 5)
binary_targets = torch.randint(0, 2, (12, 1)).float()
binary_logits = binary_model(binary_X)
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_targets)
binary_pred = torch.sigmoid(binary_logits) >= 0.5
binary_acc = (binary_pred == binary_targets.bool()).float().mean().item()
assert binary_logits.shape == binary_targets.shape and 0 <= binary_acc <= 1

# Regression: accuracy is often not meaningful; MSE/MAE/RMSE are typical.
reg_model = nn.Linear(4, 1)
reg_X = torch.randn(12, 4)
reg_y = torch.randn(12, 1)
reg_pred = reg_model(reg_X)
reg_mse = nn.MSELoss()(reg_pred, reg_y)
reg_mae = (reg_pred - reg_y).abs().mean()
reg_rmse = reg_mse.sqrt()
print(f"binary acc={binary_acc:.3f}; regression MAE={reg_mae.item():.3f}")


## 23. Common Mistakes — 🔥 Interview Essential

1. Forgetting `model.train()`, especially with Dropout/BatchNorm.
2. Forgetting `model.eval()`.
3. Forgetting `torch.no_grad()` during validation.
4. Calling `backward()` in validation.
5. Calling `optimizer.step()` in validation.
6. Forgetting `optimizer.zero_grad()`.
7. Calling `step()` before `backward()`.
8. Forgetting device transfer.
9. Putting model and tensors on different devices.
10. Averaging mean-reduced batch losses incorrectly.
11. Dividing accuracy by batches, not samples.
12. Calling `.item()` and trying to backpropagate the Python number.
13. Taking `argmax` over the wrong dimension.
14. Forgetting to switch back to training mode after validation.
15. Clearing gradients after `backward()` but before `step()`.
16. Applying softmax before `CrossEntropyLoss`.
17. Updating parameters from validation data.
18. Training indefinitely without tracking validation performance.


## 24. Debugging Exercises — 20 drills

Diagnose each before opening its answer.


### Debug 1 · 🟢 Easy

**Symptom/code:** Validation calls `loss.backward()`.

<details><summary>What is wrong?</summary>

Validation should not backpropagate; use `model.eval()` and `torch.no_grad()`.

</details>


### Debug 2 · 🟢 Easy

**Symptom/code:** `backward(); zero_grad(); step()`

<details><summary>What is wrong?</summary>

The second call clears gradients before the optimizer reads them.

</details>


### Debug 3 · 🟢 Easy

**Symptom/code:** Model is on CUDA; input remains on CPU.

<details><summary>What is wrong?</summary>

Move inputs, and usually targets, to the same device.

</details>


### Debug 4 · 🟢 Easy

**Symptom/code:** `total_loss += loss.item(); epoch_loss = total_loss / len(dataset)`

<details><summary>What is wrong?</summary>

Batch means were summed, so dividing by samples is inconsistent; weight each mean by batch size.

</details>


### Debug 5 · 🟢 Easy

**Symptom/code:** The loop never calls `zero_grad()`.

<details><summary>What is wrong?</summary>

Gradients accumulate across ordinary mini-batches.

</details>


### Debug 6 · 🟢 Easy

**Symptom/code:** The loop calls `optimizer.step()` before `loss.backward()`.

<details><summary>What is wrong?</summary>

The optimizer has no current gradients to use.

</details>


### Debug 7 · 🟢 Easy

**Symptom/code:** Validation uses `model.train()`.

<details><summary>What is wrong?</summary>

Dropout and BatchNorm remain in training behavior.

</details>


### Debug 8 · 🟢 Easy

**Symptom/code:** Training resumes after validation without `model.train()`.

<details><summary>What is wrong?</summary>

The model remains in evaluation behavior.

</details>


### Debug 9 · 🟢 Easy

**Symptom/code:** Accuracy is `correct / len(loader)`.

<details><summary>What is wrong?</summary>

Divide by number of samples, not batches.

</details>


### Debug 10 · 🟢 Easy

**Symptom/code:** Multiclass prediction uses `argmax(dim=0)`.

<details><summary>What is wrong?</summary>

Class scores are along dimension 1 for `(batch, classes)`.

</details>


### Debug 11 · 🟢 Easy

**Symptom/code:** Code applies `softmax` before `CrossEntropyLoss`.

<details><summary>What is wrong?</summary>

Pass raw logits; CrossEntropyLoss performs the stable log-softmax internally.

</details>


### Debug 12 · 🟢 Easy

**Symptom/code:** `loss_value = loss.item(); loss_value.backward()`

<details><summary>What is wrong?</summary>

A Python float has no computation graph or `backward`.

</details>


### Debug 13 · 🟡 Medium

**Symptom/code:** Only `model.eval()` is used during validation.

<details><summary>What is wrong?</summary>

Layer behavior is correct, but Autograd still tracks operations; add `no_grad()`.

</details>


### Debug 14 · 🟡 Medium

**Symptom/code:** Only `torch.no_grad()` is used during validation.

<details><summary>What is wrong?</summary>

Gradients are off, but Dropout/BatchNorm remain in their current mode; add `eval()`.

</details>


### Debug 15 · 🟡 Medium

**Symptom/code:** Targets for CrossEntropyLoss are float one-hot vectors in this basic example.

<details><summary>What is wrong?</summary>

Use integer class indices of dtype `long` for the standard target format.

</details>


### Debug 16 · 🟡 Medium

**Symptom/code:** Binary predictions threshold raw logits at 0.5.

<details><summary>What is wrong?</summary>

Apply sigmoid then threshold at 0.5, or equivalently threshold logits at 0.

</details>


### Debug 17 · 🟡 Medium

**Symptom/code:** Gradient clipping is called before `backward()`.

<details><summary>What is wrong?</summary>

Gradients do not exist yet; clip after backward and before step.

</details>


### Debug 18 · 🟡 Medium

**Symptom/code:** Accumulation calls `step()` each mini-batch.

<details><summary>What is wrong?</summary>

That is ordinary training, not accumulation; delay the update to the boundary.

</details>


### Debug 19 · 🟡 Medium

**Symptom/code:** Accumulation ignores the final incomplete window.

<details><summary>What is wrong?</summary>

Perform a final step when the loop reaches its last batch.

</details>


### Debug 20 · 🟡 Medium

**Symptom/code:** Validation accuracy changes model weights.

<details><summary>What is wrong?</summary>

The validation path is performing an update or mutating state; remove optimizer/backward and use eval/no_grad.

</details>


## 25. Fill in the Blanks — 10 active-recall drills

Write the missing tokens without running anything.


1. `model._____()` before training
2. `optimizer._____()` before forward/backward
3. `loss._____()` to calculate gradients
4. `optimizer._____()` to update parameters
5. `model._____()` before validation
6. `with torch._____():` during validation
7. `pred = logits._____(dim=1)`
8. `X_batch = X_batch._____(device)`
9. `total_loss += loss.item() * _____`
10. `accuracy = total_correct / _____`


<details><summary>Answers</summary>

1. `train`, 2. `zero_grad`, 3. `backward`, 4. `step`, 5. `eval`, 6. `no_grad`, 7. `argmax`, 8. `to`, 9. `batch_size`, 10. `total_samples`

</details>


## 26. Interview Questions — 37

Answer aloud in 20–40 seconds, then expand the panel.


### 1. What is a training loop?

<details><summary>Short Interview Answer</summary>

A repeated forward-loss-backward-update process over batches.

</details>

<details><summary>Detailed Explanation</summary>

It iterates over mini-batches, computes predictions and loss, uses Autograd to calculate gradients, and asks the optimizer to update parameters.

</details>


### 2. What is an epoch?

<details><summary>Short Interview Answer</summary>

One complete pass through the training dataset.

</details>

<details><summary>Detailed Explanation</summary>

Its number of batches depends on dataset size, batch size, and remainder handling.

</details>


### 3. What is a batch?

<details><summary>Short Interview Answer</summary>

A subset of examples processed together.

</details>

<details><summary>Detailed Explanation</summary>

Mini-batches trade off memory, throughput, and gradient noise.

</details>


### 4. Why use mini-batches?

<details><summary>Short Interview Answer</summary>

They are memory-efficient and provide efficient vectorized computation.

</details>

<details><summary>Detailed Explanation</summary>

They also yield more frequent, somewhat noisy updates than full-batch training.

</details>


### 5. What does `model.train()` do?

<details><summary>Short Interview Answer</summary>

It enables training behavior for modules such as Dropout and BatchNorm.

</details>

<details><summary>Detailed Explanation</summary>

It recursively sets the module training flag; it does not run forward, backward, or update parameters.

</details>


### 6. Does `model.train()` start training automatically?

<details><summary>Short Interview Answer</summary>

No.

</details>

<details><summary>Detailed Explanation</summary>

Learning still requires a forward pass, loss, backward pass, and optimizer step.

</details>


### 7. What does `model.eval()` do?

<details><summary>Short Interview Answer</summary>

It enables evaluation behavior for stateful modules.

</details>

<details><summary>Detailed Explanation</summary>

Dropout is disabled and BatchNorm uses stored running statistics; Autograd remains enabled unless separately disabled.

</details>


### 8. `model.train()` vs `model.eval()`?

<details><summary>Short Interview Answer</summary>

They select training versus evaluation module behavior.

</details>

<details><summary>Detailed Explanation</summary>

The difference matters for layers such as Dropout and BatchNorm, not for whether an optimizer runs.

</details>


### 9. What does `torch.no_grad()` do?

<details><summary>Short Interview Answer</summary>

It disables gradient tracking inside its context.

</details>

<details><summary>Detailed Explanation</summary>

This avoids building an unnecessary graph and generally reduces validation/inference memory and computation.

</details>


### 10. `model.eval()` vs `torch.no_grad()`?

<details><summary>Short Interview Answer</summary>

Mode behavior versus Autograd tracking.

</details>

<details><summary>Detailed Explanation</summary>

They solve different problems, so validation normally uses both.

</details>


### 11. Why call `optimizer.zero_grad()`?

<details><summary>Short Interview Answer</summary>

PyTorch gradients accumulate by default.

</details>

<details><summary>Detailed Explanation</summary>

Clearing prevents a new ordinary mini-batch gradient from being added to old gradients.

</details>


### 12. What does `loss.backward()` do?

<details><summary>Short Interview Answer</summary>

It computes gradients of loss with respect to graph leaves such as parameters.

</details>

<details><summary>Detailed Explanation</summary>

The results accumulate in each parameter's `.grad` field.

</details>


### 13. What does `optimizer.step()` do?

<details><summary>Short Interview Answer</summary>

It updates parameters using their current gradients.

</details>

<details><summary>Detailed Explanation</summary>

The exact rule depends on the optimizer, its hyperparameters, and state.

</details>


### 14. What is the correct training-step order?

<details><summary>Short Interview Answer</summary>

zero_grad → forward → loss → backward → step.

</details>

<details><summary>Detailed Explanation</summary>

Each stage creates information needed by the next.

</details>


### 15. Why should validation not call backward?

<details><summary>Short Interview Answer</summary>

Validation measures performance; it does not optimize.

</details>

<details><summary>Detailed Explanation</summary>

Backpropagation wastes resources and can accidentally support unwanted updates.

</details>


### 16. Why should validation not update parameters?

<details><summary>Short Interview Answer</summary>

It would leak validation information into training.

</details>

<details><summary>Detailed Explanation</summary>

The validation set must remain an unbiased model-selection signal.

</details>


### 17. Why use `no_grad()` during validation?

<details><summary>Short Interview Answer</summary>

To avoid unnecessary graph construction.

</details>

<details><summary>Detailed Explanation</summary>

It usually reduces memory and overhead while ensuring no gradient graph is retained.

</details>


### 18. How do you calculate multiclass predictions?

<details><summary>Short Interview Answer</summary>

`logits.argmax(dim=1)`.

</details>

<details><summary>Detailed Explanation</summary>

For logits shaped `(batch, classes)`, dimension 1 is the class dimension.

</details>


### 19. How do you calculate classification accuracy?

<details><summary>Short Interview Answer</summary>

Correct predictions divided by total samples.

</details>

<details><summary>Detailed Explanation</summary>

Accumulate integer correct counts and sample counts across all batches before dividing.

</details>


### 20. How do you average loss across unequal batches?

<details><summary>Short Interview Answer</summary>

Weight each batch mean by its batch size, sum, then divide by samples.

</details>

<details><summary>Detailed Explanation</summary>

This reconstructs the dataset-level mean when the loss uses mean reduction.

</details>


### 21. Why multiply mean loss by batch size?

<details><summary>Short Interview Answer</summary>

To recover the batch's total loss contribution.

</details>

<details><summary>Detailed Explanation</summary>

Otherwise a small final batch receives the same weight as a full batch.

</details>


### 22. What does `loss.item()` do?

<details><summary>Short Interview Answer</summary>

It returns the scalar tensor value as a Python number.

</details>

<details><summary>Detailed Explanation</summary>

Use it for logs and metric accumulation, not for backpropagation.

</details>


### 23. Why not use `.item()` for backpropagation?

<details><summary>Short Interview Answer</summary>

It detaches the numeric value from the graph.

</details>

<details><summary>Detailed Explanation</summary>

The returned Python number has no `backward()` method or graph history.

</details>


### 24. Why must model and input share a device?

<details><summary>Short Interview Answer</summary>

Operations generally require participating tensors on the same device.

</details>

<details><summary>Detailed Explanation</summary>

A CPU/CUDA mismatch raises a runtime error.

</details>


### 25. How do you move a model to GPU?

<details><summary>Short Interview Answer</summary>

`model.to(device)` where device is CUDA.

</details>

<details><summary>Detailed Explanation</summary>

Move every input and target used by device operations as well.

</details>


### 26. How do you move a batch to GPU?

<details><summary>Short Interview Answer</summary>

`X, y = X.to(device), y.to(device)`.

</details>

<details><summary>Detailed Explanation</summary>

DataLoaders normally yield CPU tensors unless a specialized pipeline says otherwise.

</details>


### 27. What is gradient clipping?

<details><summary>Short Interview Answer</summary>

Limiting gradient magnitude before an optimizer update.

</details>

<details><summary>Detailed Explanation</summary>

Norm clipping rescales gradients whose combined norm exceeds a threshold.

</details>


### 28. When is gradient clipping useful?

<details><summary>Short Interview Answer</summary>

When gradients explode or training is unstable.

</details>

<details><summary>Detailed Explanation</summary>

It is common in recurrent networks and some large sequence models, but does not fix every instability.

</details>


### 29. What is gradient accumulation?

<details><summary>Short Interview Answer</summary>

Summing gradients across multiple mini-batches before stepping.

</details>

<details><summary>Detailed Explanation</summary>

It approximates a larger effective batch under memory constraints.

</details>


### 30. Why divide loss by accumulation steps?

<details><summary>Short Interview Answer</summary>

To keep gradient scale near that of a larger averaged batch.

</details>

<details><summary>Detailed Explanation</summary>

Without scaling, the summed gradient is roughly multiplied by the number of accumulation steps.

</details>


### 31. What is effective batch size?

<details><summary>Short Interview Answer</summary>

Approximately physical batch × accumulation steps × data-parallel devices.

</details>

<details><summary>Detailed Explanation</summary>

That approximation assumes standard synchronous accumulation.

</details>


### 32. What is early stopping?

<details><summary>Short Interview Answer</summary>

Stopping when validation performance fails to improve.

</details>

<details><summary>Detailed Explanation</summary>

It is a regularization and resource-saving strategy, usually paired with best-checkpoint restoration.

</details>


### 33. What does patience mean?

<details><summary>Short Interview Answer</summary>

How many non-improving epochs are tolerated.

</details>

<details><summary>Detailed Explanation</summary>

After patience is exhausted, training stops according to the chosen improvement rule.

</details>


### 34. What can train vs validation loss show?

<details><summary>Short Interview Answer</summary>

A widening gap can suggest overfitting.

</details>

<details><summary>Detailed Explanation</summary>

High values for both can suggest underfitting, while noisy/exploding curves may suggest optimization problems.

</details>


### 35. What could cause loss to explode?

<details><summary>Short Interview Answer</summary>

An excessive learning rate or exploding gradients, among other causes.

</details>

<details><summary>Detailed Explanation</summary>

Also inspect bad data scales, numerical instability, wrong loss/targets, and faulty loop order.

</details>


### 36. What could cause training to be extremely slow?

<details><summary>Short Interview Answer</summary>

Large data/model, device bottlenecks, or unnecessary graph work.

</details>

<details><summary>Detailed Explanation</summary>

Also inspect data loading, CPU/GPU transfer, synchronization, logging, and validation without no_grad.

</details>


### 37. Why set a random seed?

<details><summary>Short Interview Answer</summary>

To make randomized experiments more reproducible.

</details>

<details><summary>Detailed Explanation</summary>

It controls random-number streams but does not by itself guarantee cross-platform determinism.

</details>


## 27. Knowledge Check Quiz — 30 questions

Choose one answer per question before opening the answer key.


**1. Normal training order?**

A. forward → step → backward

B. zero_grad → forward → loss → backward → step

C. backward → zero_grad → step

D. step → forward → loss


**2. What does `model.eval()` do?**

A. Disables Autograd

B. Updates parameters

C. Changes Dropout/BatchNorm behavior

D. Clears gradients


**3. What does `torch.no_grad()` do?**

A. Switches Dropout off

B. Disables gradient tracking

C. Clears gradients

D. Updates parameters


**4. An epoch is...**

A. one sample

B. one batch

C. one full dataset pass

D. one optimizer object


**5. Why zero gradients?**

A. Gradients accumulate

B. Loss is a float

C. Data are shuffled

D. Eval needs it


**6. Multiclass prediction from `(B,C)` logits?**

A. argmax(0)

B. mean(1)

C. argmax(1)

D. sigmoid only


**7. Accuracy denominator?**

A. classes

B. batches

C. parameters

D. samples


**8. When clip gradients?**

A. before forward

B. after backward, before step

C. after step

D. during eval


**9. Validation normally uses...**

A. train + backward

B. eval + no_grad

C. eval + step

D. train + step


**10. `loss.item()` returns...**

A. parameter

B. Python scalar

C. gradient graph

D. optimizer


**11. Mean batch loss should be weighted by...**

A. class count

B. epoch index

C. batch size

D. learning rate


**12. `model.train()`...**

A. starts training automatically

B. sets module mode

C. creates optimizer

D. loads data


**13. `model.eval()` alone...**

A. disables gradients

B. does not disable gradients

C. updates weights

D. clears state


**14. CrossEntropyLoss expects primary example input as...**

A. probabilities

B. raw logits

C. argmax labels

D. thresholds


**15. Gradient accumulation is used mainly to...**

A. delete gradients

B. simulate larger effective batch

C. disable Dropout

D. split classes


**16. Why scale accumulated loss?**

A. change labels

B. approximate averaged-batch gradient scale

C. enable eval

D. move device


**17. Physical batch 16, accumulation 4 gives...**

A. 4

B. 16

C. 20

D. about 64


**18. Patience belongs to...**

A. clipping

B. early stopping

C. DataLoader

D. CrossEntropy


**19. Overfitting may show...**

A. train loss down, val loss up

B. both exactly zero

C. no batches

D. no logits


**20. Underfitting may show...**

A. both losses remain high

B. train down and val up

C. perfect validation

D. no optimizer


**21. Model CUDA + input CPU causes...**

A. automatic transfer

B. device mismatch

C. faster training

D. Dropout


**22. Binary loss commonly used?**

A. MSE only

B. BCEWithLogitsLoss

C. CrossEntropy with one raw output only

D. MAE only


**23. Regression prediction normally uses...**

A. raw output

B. argmax

C. class threshold

D. one-hot


**24. Backward stores gradients in...**

A. DataLoader

B. parameter `.grad`

C. history dict

D. target


**25. Optimizer step reads...**

A. current gradients

B. validation labels

C. plot

D. batch count only


**26. Final small batch matters most for...**

A. sample-weighted loss

B. model mode

C. device selection

D. seed syntax


**27. Validation should update parameters?**

A. always

B. only last batch

C. no

D. only with no_grad


**28. Dropout during eval is normally...**

A. disabled

B. doubled

C. optimized

D. a loss


**29. Seed guarantees all hardware is bitwise identical?**

A. yes

B. no

C. only without model

D. only batch 1


**30. Final partial accumulation window needs...**

A. discarding always

B. a final optimizer step

C. eval mode

D. softmax


### Quiz answer key

1. **B** · 2. **C** · 3. **B** · 4. **C** · 5. **A** · 6. **C** · 7. **D** · 8. **B** · 9. **B** · 10. **B** · 11. **C** · 12. **B** · 13. **B** · 14. **B** · 15. **B** · 16. **B** · 17. **D** · 18. **B** · 19. **A** · 20. **A** · 21. **B** · 22. **B** · 23. **A** · 24. **B** · 25. **A** · 26. **A** · 27. **C** · 28. **A** · 29. **B** · 30. **B**


## 28. Write From Memory — 20 prompts

Do these in a fresh cell or on paper. Compare with the functions and cheat sheet only after finishing.

1. Write one complete training step.
2. Put a model into training mode and explain what changes.
3. Put a model into evaluation mode.
4. Disable gradient tracking for validation.
5. Calculate multiclass predictions.
6. Calculate sample-level classification accuracy.
7. Move a model to `device`.
8. Move input and target to `device`.
9. Write a complete one-epoch training loop.
10. Write a complete validation loop.
11. Accumulate sample-weighted average loss.
12. Add gradient clipping in the correct position.
13. Write gradient accumulation for four mini-batches.
14. Handle the final partial accumulation window.
15. Write a multi-epoch train + validation loop.
16. Create and update a four-metric history dictionary.
17. Write a binary prediction conversion.
18. Write a regression loss step.
19. Write early-stopping pseudocode with patience 3.
20. Explain the entire loop aloud in 45 seconds.


### Practical self-check harness

Paste your own `my_train_one_epoch` above this cell, then adapt the commented checks. A correct function changes at least one parameter and returns valid Python metrics.


In [ ]:
# Example validation pattern for your from-memory implementation:
# before = [p.detach().clone() for p in my_model.parameters()]
# my_loss, my_acc = my_train_one_epoch(...)
# after = list(my_model.parameters())
# assert any(not torch.equal(a, b) for a, b in zip(before, after))
# assert isinstance(my_loss, float)
# assert 0 <= my_acc <= 1
print("self-check template ready")


## 29. 10-Minute Training Loop Challenge

Without looking up, replace every `...` in a new cell. Requirements: modes, device transfer, zero gradients, forward, loss, backward, update, `no_grad`, weighted loss, and accuracy.

```python
for epoch in range(num_epochs):
    # TRAIN
    ...

    # VALIDATE
    ...
```

Success criteria: both losses are Python floats, both accuracies are in `[0,1]`, training parameters change, validation parameters do not.


## 30. Code Reading Exercise

Explain each block verbally as if answering an interview:

```python
for epoch in range(10):
    model.train()
    for X, y in train_loader:
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        for X, y in val_loader:
            pred = model(X)
            val_loss = loss_fn(pred, y)
```

<details><summary>Concise model answer</summary>

The outer loop repeats epochs. Training mode is enabled, then every training batch clears old gradients, runs forward, computes loss, backpropagates, and updates parameters. Evaluation mode then changes Dropout/BatchNorm behavior; `no_grad` prevents graph construction while validation batches run forward and compute loss without updates.

</details>


## 31. Final Training Challenge

Build the whole task from memory before reading the reference implementation below.

- 1,200 samples, 20 features, 4 classes; tensor-only 80/20 split.
- Model: `20 → 64 → ReLU → 32 → ReLU → 4 logits`.
- CUDA, else MPS, else CPU.
- Cross-entropy and AdamW (`lr=1e-3`, `weight_decay=1e-2`).
- Implement train/evaluate, run 10–20 epochs, log and store all four metrics.
- Assert both accuracies are valid and verify meaningful learning.


In [ ]:
torch.manual_seed(42)
num_samples, num_features, num_classes = 1200, 20, 4
challenge_X = torch.randn(num_samples, num_features)
true_weights = torch.randn(num_features, num_classes)
scores = challenge_X @ true_weights
challenge_y = scores.argmax(dim=1)

split = int(0.8 * num_samples)
perm = torch.randperm(num_samples)
train_idx, val_idx = perm[:split], perm[split:]
challenge_train = TensorDataset(challenge_X[train_idx], challenge_y[train_idx])
challenge_val = TensorDataset(challenge_X[val_idx], challenge_y[val_idx])
challenge_train_loader = DataLoader(challenge_train, batch_size=64, shuffle=True)
challenge_val_loader = DataLoader(challenge_val, batch_size=64, shuffle=False)

challenge_model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 4),
).to(device)
challenge_loss_fn = nn.CrossEntropyLoss()
challenge_optimizer = optim.AdamW(
    challenge_model.parameters(), lr=1e-3, weight_decay=1e-2
)

challenge_history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
challenge_epochs = 12

for epoch in range(challenge_epochs):
    tr_loss, tr_acc = train_one_epoch(
        challenge_model, challenge_train_loader, challenge_loss_fn,
        challenge_optimizer, device,
    )
    va_loss, va_acc = evaluate(
        challenge_model, challenge_val_loader, challenge_loss_fn, device
    )
    challenge_history["train_loss"].append(tr_loss)
    challenge_history["val_loss"].append(va_loss)
    challenge_history["train_acc"].append(tr_acc)
    challenge_history["val_acc"].append(va_acc)
    print(
        f"Epoch {epoch + 1:02d} | Train Loss {tr_loss:.4f} | Train Acc {tr_acc:.3f} | "
        f"Val Loss {va_loss:.4f} | Val Acc {va_acc:.3f}"
    )

assert 0 <= tr_acc <= 1 and 0 <= va_acc <= 1
assert challenge_history["train_loss"][-1] < challenge_history["train_loss"][0]
assert challenge_history["train_acc"][-1] > challenge_history["train_acc"][0]
print("Final challenge learned meaningfully ✓")


### Optional Harder Challenge — 🔴 Hard

Add `torch.nn.utils.clip_grad_norm_` after backward and before step. Then implement early stopping with patience, keep a clone of the best `state_dict`, and restore it after stopping.


## 32. Final Cheat Sheet

```python
# Training
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

# Validation
model.eval()
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)

# Metrics
pred = logits.argmax(dim=1)
correct = (pred == y).sum().item()
batch_size = y.size(0)
total_loss += loss.item() * batch_size
avg_loss = total_loss / total_samples
accuracy = total_correct / total_samples

# Gradient clipping
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```


## 33. Critical Mental Model and Interview Quick Reference

```text
TRAINING
model.train() → batch → zero_grad → forward → loss → backward → step

VALIDATION
model.eval() → torch.no_grad() → batch → forward → loss + metrics
                                      NO backward · NO step

model.eval() ≠ torch.no_grad()
```

```text
epoch                 → one pass through dataset
batch                 → subset of dataset
model.train()         → training behavior
model.eval()          → evaluation behavior
torch.no_grad()       → disable Autograd
zero_grad()           → clear previous gradients
backward()            → calculate gradients
step()                → update parameters
loss.item()           → scalar Python value
gradient clipping     → limit gradient magnitude
gradient accumulation → combine gradients across mini-batches
early stopping        → stop when validation stops improving
```


## Before Moving to 07_dataset_dataloader.ipynb

- [ ] I can explain epoch vs batch.
- [ ] I can write one training step from memory.
- [ ] I can write a full training loop from memory.
- [ ] I understand `model.train()`.
- [ ] I understand `model.eval()`.
- [ ] I understand `torch.no_grad()`.
- [ ] I can explain `model.eval()` vs `torch.no_grad()`.
- [ ] I can calculate training and validation loss.
- [ ] I can calculate multiclass accuracy.
- [ ] I can average batch loss correctly.
- [ ] I can move model and batches to a device.
- [ ] I can train for multiple epochs and log metrics.
- [ ] I understand learning curves, clipping, accumulation, and early stopping.
- [ ] I can debug common loop mistakes.
- [ ] I can implement `train_one_epoch()` and `evaluate()`.
